# Layer 4: Field Planning (class notes)

**What this layer is, in one line:** it decides where to send the field officers so that the village calls become right as often as possible.

You do not need to know how the code is written. You need to be able to explain each idea, why it is needed and how it leads to the next. **Everything in this layer is a computer simulation.** There was no real field campaign.

## The story of this layer in one paragraph

Layer 3 gives every village a call and a probability, and some calls are shaky (section 1). Officers can only do a limited amount, so the question is where a check does the most good. We define "does the most good" as "most likely to turn a wrong call into a right one" (section 2). In practice officers must also drive, and floods slow or close roads, so we build a road map that knows about floods (section 3) and then plan whole days of driving, balancing payoff against time (section 4). Routes naturally favour easy places, which makes a biased sample, so we add a small random set of villages with known chances (section 5). Finally we test every choice fairly (section 6) and hand a plan to the tool (section 7).

## 1. The goal

*Where we are: Layer 3 has produced a call and a probability for every village.*

Some calls are shaky: the estimate is close to the 25 percent line. We have only **3 officers and 4 hours a day**, and some roads are flooded. Layer 4 answers three questions:

| Question | Short answer |
|---|---|
| Which check should come next? | The one most likely to turn a wrong call into a right one |
| Which villages and roads today? | The best payoff per extra minute of driving, on a map that knows about flooded roads |
| How do we keep the sample fair? | Visit a few villages picked at random with known chances |

## 2. Which check next?

*Where we are: we have a budget of checks. Where should each one go?*

**The chance a call is wrong.** Each village's estimate has an uncertainty. If the estimate is very close to the 25 percent line, the call could easily be wrong. If it is far from the line, the call is almost surely right, even if the estimate is rough. What matters is how far the estimate is from the line **measured in units of its uncertainty**.

**The rule.** Send the next check where it is **expected to cut the chance of a wrong call the most**. To judge this we imagine what a check might find. It would nudge the estimate in some direction and make it more certain. We average over all the nudges it might cause and compare the chance of being wrong before and after.

**Worked example (made-up numbers).** Three villages, with the 25 percent line:

| Village | Estimate | Uncertainty | Chance call is wrong |
|---|---|---|---|
| A | 0.26 | 0.06 | about 43 percent (right on the line) |
| B | 0.40 | 0.06 | under 1 percent (clearly above) |
| C | 0.10 | 0.10 | about 7 percent (far below, but noisy) |

A rule that says "check where we are least certain" picks **C**, which has the biggest uncertainty. But C is already far below the line and is almost surely a correct "below". Our rule picks **A**, because one more check could easily move A clearly to one side. That is the whole idea: **spend checks where they can change the answer.**

**Does it work?** Simulation with 80 villages and 60 test scenarios, wrong calls at **16 checks per village**:

| Rule | Wrong calls |
|---|---|
| Random placement | 14.0 percent |
| Proportional to village size | 13.8 percent |
| Check the most uncertain | 12.6 percent |
| **Our rule** | **8.6 percent** |

Our rule reaches the others' 32-check accuracy with about 12 checks.

**Honest limits:**
* With only **2 to 4 checks per village**, every rule is **worse than just using the map**. Early checks are noisy and swing the estimate. The map alone gets 11.4 percent wrong; ours gets 19.2 at 2 checks and 14.7 at 4. Ours matches the map at about 8 and beats it from 12.
* On real map errors (91 test pseudo-villages) our rule is best up to 16 checks per village but **loses its edge at 32** (3.6 percent against 3.1 for "check the most uncertain"). Once most villages are settled, spending on the rest by uncertainty is as good.

*So next: this ignored travel completely. A check is only possible if an officer can reach the place.*

## 3. Roads: the flood-aware map

*Where we are: we know where checks would help. Now we need to know the cost of getting there.*

From Layer 1 we have the road map with normal travel times. From Layer 2 we have the chance that each road piece is flooded. We build a **flood-aware** map: roads that are probably flooded get slower, and roads that are very probably flooded (above **70 percent**) are treated as closed. Then the shortest-path method (Dijkstra, from Layer 1) finds the fastest way to every village.

**A surprise worth knowing.** In our simulated truth a flooded road takes **8 times** as long. You would guess the planner should use 8. It does worse than ignoring floods! The reason: on the day, an officer takes whichever route happens to be dry. The *average* slowdown therefore overstates what the officer actually experiences. So the planning slowdown was **fitted** on tuning scenarios, giving **4**. Typical error of planned travel times against the simulated truth:

| Planning map | Typical error |
|---|---|
| Ignore floods | 29.5 percent (too optimistic) |
| Use the true factor 8 | 52.1 percent (too pessimistic) |
| **Fitted factor 4** | **21.0 percent** |

*So next: with check values and travel times, we can plan a day.*

## 4. Which villages and roads today?

*Where we are: each village has a value (how much a 12-check visit reduces wrong calls) and a travel cost.*

Each morning the planner does this:

1. For every village, work out the expected drop in wrong calls from a visit (**12 checks, about 18 minutes** at 1.5 minutes per check).
2. Repeatedly pick the village with the best **gain per extra minute** and add it to the officer's route, in the cheapest position, as long as the officer's day still fits in 4 hours.
3. Officers drive the (simulated) real roads and turn back if time runs out.
4. Next morning, recompute everything with the new results.

**Why gain per extra minute?** Example (made-up): village X is worth 0.10 but needs 40 extra minutes. Village Y is worth 0.05 but only 10 extra minutes. Per minute: X gives 0.0025, Y gives 0.005. Y is the better use of time. A village on the way costs very few extra minutes.

This kind of problem is called **orienteering**: collect the most reward within a time limit. It is hard to solve perfectly, so we use a quick, greedy rule. That is good in practice, not guaranteed best.

**Compared over 5 days in simulation** (30 villages, 60 scenarios):

| Plan | Wrong calls after day 5 |
|---|---|
| Nearest village first (the usual practice) | 13.3 percent |
| Highest gain, ignore travel | 11.1 percent |
| Gain per minute, ignoring floods | 10.6 percent |
| **Gain per minute, flood-aware** | **9.7 percent** |
| Map alone (no visits) | 10.9 percent |

Flood-aware beats nearest-first by about **3.6 points**. Honest points:
* Flood-aware against flood-blind is not a statistically clear win in the simulation (it does about **27 percent less driving** for about the same result, which is its clearer benefit).
* On days 1 to 3 every plan is worse than the map alone, for the same reason as in section 2.
* On the real road network of the study block (a small area) the plan still beat nearest-first, by about 4.4 points.

*So next: the plan favours easy villages. That raises a fairness problem.*

## 5. Keeping the sample fair: the random "spine"

*Where we are: the gain-based plan keeps visiting villages that are near the depot and on dry roads.*

Those checks are not a fair sample of the whole region. This does not hurt each village's own call, but it **biases anything we average across villages**, for example how often the map is right overall.

The fix is a **spine**: a small set of villages picked **at random, with known chances**, that are always visited.

* Before day 1, each village that can be reached within a reasonable part of a day is picked by an independent weighted coin flip (**Poisson sampling**). Larger villages get a higher chance, with a floor of **5 percent**, set so that about **8 villages** are expected.
* Picked villages are visited first. Inside them, **6 plots per group** are checked by simple random selection.
* Because each plot's chance of being checked is known (village chance times plot chance within the village), the checks can later be **weighted** to remove bias (Layer 1, section 8; Layer 3, section 6).

**Limits.** The spine can only draw from villages that can be reached, which covers about **half** of all plots, and in about a quarter of simulated regions no village was reachable at all. Weighting cannot fix places that could never be picked.

**Result in simulation.** Weighted spine estimates showed no detectable bias. Pooling all the routed checks overstated one rate by about 1.5 points. But in our simulated world routing bias is small, so the spine buys a **guarantee**, not a smaller error. It also costs no measurable accuracy in the village calls.

*So next: how do we know these results are not just lucky choices of settings?*

## 6. Fair testing

*Where we are: we chose a few settings and then scored the planner.*

Settings such as the planning slowdown (4), the road-closing level (70 percent) and how to weight the spine were chosen using separate **tuning scenarios**. The results above come from **other** scenarios that were never used for tuning, so they are not flattered. We also compared every smart method against simple baselines.

**What is real and what is simulated.** The road map (OpenStreetMap) and the Layer 2 chip maps are real. The flood over the study block, the field truth, the officers and the 8-times slowdown are simulated or assumed. Plots inside a village are treated as independent in the simulation, although Layer 3 shows that real checks cluster.

## 7. What this layer hands to the tool

A day-1 plan: for each officer, the villages in order, the road path and the planned minutes (in our run, 3 officers with 8, 7 and 5 villages), plus a ranked list of next checks and the chance each village had of being drawn for the spine. In the committed tool run, the routes shown are a stand-in fixture and no Layer 4 checks are used.

## 8. How Layer 4 connects to the others

| Layer 4 uses | From |
|---|---|
| calls, uncertainties, loss line, prior strength | Layer 3 |
| road graph and travel times | Layer 1 |
| chance each road is flooded | Layer 2 |
| the weighting idea | Layer 1 and Layer 3 |

| Layer 4 produces | Used by |
|---|---|
| day-1 routes, ranked next checks, selection chances | Layer 5 |

## 9. Key numbers

| Thing | Value |
|---|---|
| Officers, day, visit | **3** officers, **4 hours**, **12 checks** (about 18 minutes) per visit |
| Campaign length | **5 days** |
| Flooded road, truth vs plan | truth **8 times** slower, plan uses **4 times** |
| Close a road above | **70 percent** flood chance |
| Spine | about **8** villages, minimum chance **5 percent**, covers about **half** of plots |
| 16 checks, wrong calls | ours **8.6**, others 12.6 to 14.0, map alone 11.4 |
| 2 to 4 checks | every rule worse than the map |
| After 5 days | flood-aware plan **9.7**, nearest first **13.3** |

## 10. Glossary

* **Greedy**: always take the best-looking next step, without planning far ahead.
* **Orienteering problem**: collect the most reward within a time limit.
* **Flood-aware**: road times adjusted by the chance of flooding.
* **Poisson sampling**: each item gets its own independent chance of being picked.
* **Spine**: a small random set of villages with known chances, always visited.
* **Tuning scenarios**: simulations used only to choose settings, never for scoring.
* **Gain**: expected drop in the chance of a wrong call.

## 11. Review questions

1. **What does Layer 4 do?** Decides where officers check and which roads they drive.
2. **How is the next check chosen?** Where it is expected to reduce the chance of a wrong call the most.
3. **Why not check the most uncertain village?** A village far from the line stays right however uncertain; we focus on villages near the line.
4. **How much better is it?** At 16 checks, 8.6 percent wrong against 12.6 to 14.0.
5. **Where does it not win?** With 2 to 4 checks every rule is worse than the map, and at 32 checks on real errors it loses its edge.
6. **Why does the plan assume 4 times, not 8?** Officers take whichever road is dry, so 8 overstates travel by about 50 percent.
7. **How are routes built?** Greedily, by best gain per extra minute, re-planned each morning.
8. **What is the spine and why?** A random set of villages with known chances, so checks can be weighted to remove routing bias.
9. **Does the spine cost accuracy?** No measurable loss, but its bias fix is a guarantee more than a big gain in this simulation.
10. **Is any of this real?** Roads and chip maps are real; the field work, flooding over the block and the slowdown are simulated or assumed.